# 42 · Can we predict exceptional winners, not merely average returns?

**Development-only research.** Notebook 41 found weak overall ranking IC, tail-dependent profits, and a large discrepancy between earlier Ridge results. This notebook first reconstructs the *identical Notebook 41 Ridge baseline* before testing event probabilities, conditional return quantiles, two-stage ranking, risk-adjusted ranking, concentration and paired comparisons. **No final historical holdout or prospective prediction ledger is accessed.** Zero transaction costs; no hypothetical stop-loss/take-profit assumptions. All event labels are defined from training outcomes; validation outcomes are used only for evaluation. This is exploratory research after many prior model comparisons.

In [ ]:
from pathlib import Path
import sys,time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir(),f'Run from repository root or notebooks/, not {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.tail_edge_42 import *
from stock_ml_dynamic.edge_robustness_41 import fit_and_score,run_portfolio_stress
OUT=ROOT/'artifacts/tail_edge/1d';OUT.mkdir(parents=True,exist_ok=True)
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'Inference rows:',latest.shape,'Output:',OUT)

## Experiments 1–3 · Recreate the development universe and freeze chronological folds

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,features=add_volume_regime_features(expanded)
print('Development:',expanded.shape,'Feature extras:',len(features),'Holdout cutoff:',cutoff)
assert len(expanded)>0 and len(features)>0

## Experiments 4–6 · **Baseline reconciliation gate**

First reconstruct Notebook 41's Ridge α100 independently using its original function, with the same 60,000-row cap and seed. This avoids comparing incompatible baselines. If the two implementations disagree beyond numerical tolerance, **stop and debug before drawing any model conclusions**. Notebook 39's +0.262% and Notebook 41's fold-averaged +0.148% are *not assumed equivalent*; their pipelines, fold dates, and aggregation must be audited separately.

In [ ]:
import time
start=time.perf_counter()
old_scored,old_coefs,old_audit=fit_and_score(expanded,features,cap=60000,seed=42)
old_result=run_portfolio_stress(old_scored)
old_baseline=old_result['metrics'].loc[(old_result['metrics']['model']=='ridge_all') & (old_result['metrics']['k']==5)].sort_values('fold')
print('Notebook 41 reconstructed baseline (mean of fold-level daily means):',old_baseline.mean_daily.mean())
display(old_baseline[['fold','days','mean_daily','market_excess','payoff_ratio','profit_factor']])
print('Baseline reconstruction seconds:',round(time.perf_counter()-start,1))

## Experiments 7–18 · Tail-event classification at +3%, +5%, +10%: Logistic vs Histogram Gradient Boosting

In [ ]:
start=time.perf_counter()
results=run_tail_research(expanded,features,cap=60000,seed=42)
scored,metrics,daily,trades=results['scored'],results['metrics'],results['daily'],results['trades']
print('Experiment time seconds:',round(time.perf_counter()-start,1))
display(results['audit'])
print('Score variants:',len([c for c in scored if c not in ('bar_timestamp','symbol','next_return','fold')]))
display(results['events'].round(5))

## Experiment 19 · Hard baseline equivalence assertion (Notebook 41 vs Notebook 42)

In [ ]:
base42=metrics.loc[(metrics['model']=='ridge_all') & (metrics['k']==5)].sort_values('fold')
compare=old_baseline[['fold','mean_daily']].merge(base42[['fold','mean_daily']],on='fold',suffixes=('_41','_42'))
compare['difference']=compare.mean_daily_42-compare.mean_daily_41
display(compare)
assert len(compare)==3 and np.allclose(compare.mean_daily_41,compare.mean_daily_42,atol=1e-9), 'STOP: Ridge baseline differs between Notebook 41 and 42.'
print('PASS: Notebook 42 reproduces the Notebook 41 Ridge baseline exactly within tolerance.')

## Experiments 20–23 · Tail-event prevalence, discrimination, precision and Brier calibration

In [ ]:
ev=results['events'];display(ev[ev.threshold.notna()].sort_values(['threshold','model','fold']).round(6))
print('Mean event discrimination across folds:')
display(ev[ev.threshold.notna()].groupby(['threshold','model'],as_index=False).agg(auc=('auc','mean'),average_precision=('average_precision','mean'),brier=('brier','mean'),prevalence=('validation_prevalence','mean')).round(5))

## Experiments 24–26 · Probability reliability, fixed calibration bins and fold stability

In [ ]:
cal=event_calibration(scored,threshold=.05,bins=10)
display(cal.groupby(['model','bin'],as_index=False).agg(n=('n','sum'),predicted=('predicted','mean'),observed=('observed','mean')).round(5))
for name,g in cal.groupby('model'):
    a=g.groupby('bin').agg(predicted=('predicted','mean'),observed=('observed','mean'),n=('n','sum'))
    fig,ax=plt.subplots(figsize=(5,4));ax.plot([0,1],[0,1],'--',alpha=.4);ax.scatter(a.predicted,a.observed,s=np.maximum(a.n/10,5));ax.set(xlabel='Mean predicted probability',ylabel='Observed +5% frequency',title=f'{name} reliability');plt.show()

## Experiments 27–35 · Conditional quantile models: q10, q50, q90, downside-adjusted and upside-focused rankings

In [ ]:
leader=metrics.groupby(['model','k'],as_index=False).agg(mean_daily=('mean_daily','mean'),market_excess=('market_excess','mean'),payoff_ratio=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'),positive_folds=('mean_daily',lambda z:int((z>0).sum())),worst_drawdown=('max_drawdown','min'))
for k in (5,10,20):
    print(f'\nTOP 20 MODELS AT k={k}')
    display(leader.loc[leader['k']==k].sort_values('mean_daily',ascending=False).head(20).round(6))

## Experiments 36–39 · Two-stage return × tail probability and return-per-risk rankings

In [ ]:
candidates=['ridge_all','two_stage_ridge_x_p5','two_stage_ridge_x_hgb5','upside_minus_downside','upside_downside_ratio','quantile_midpoint','quantile_spread','ridge_per_spread','ridge_plus_tail_prob']
display(leader.loc[leader.model.isin(candidates) & leader.k.eq(5)].sort_values('mean_daily',ascending=False).round(6))
print('Fold-level mean daily returns:')
display(metrics.loc[metrics.model.isin(candidates) & metrics.k.eq(5)].pivot(index='model',columns='fold',values='mean_daily').round(6))

## Experiments 40–43 · Win rate, 2:1 payoff threshold, profit factor and drawdowns

In [ ]:
display(metrics.loc[metrics.k.eq(5),['model','fold','win_rate','avg_win','avg_loss','payoff_ratio','profit_factor','mean_daily','max_drawdown','meets_2_to_1']].sort_values(['fold','mean_daily'],ascending=[True,False]).round(5))
print('Count of 2:1 payoff observations by model across 3 folds:')
display(metrics.loc[metrics.k.eq(5)].groupby('model').agg(folds_2to1=('meets_2_to_1','sum'),mean_payoff=('payoff_ratio','mean')).sort_values('mean_payoff',ascending=False).round(4))

## Experiments 44–46 · Paired 10-day moving-block bootstrap against the *reconciled* Ridge baseline

In [ ]:
paired=compare_to_ridge(daily,k=5,reps=1200,seed=42)
paired_summary=paired.groupby('model',as_index=False).agg(mean_lift=('mean_lift','mean'),positive_fold_lifts=('mean_lift',lambda x:int((x>0).sum())),ci_excludes_zero=('ci_low',lambda x:0))
paired_summary['all_fold_ci_positive']=paired.groupby('model')['ci_low'].apply(lambda z:bool((z>0).all())).to_numpy()
display(paired_summary.sort_values('mean_lift',ascending=False).round(6))
print('Top challenger fold-specific CIs:')
top=paired_summary.sort_values('mean_lift',ascending=False).head(6).model.tolist()
display(paired.loc[paired.model.isin(top)].sort_values(['model','fold']).round(6))

## Experiments 47–50 · Tail dependence, top contributors, fold-wise concentration and hindsight stress

These remove the largest realized winners **only as diagnostics**; they are not valid ex-ante trading rules. A high positive-tail event AUC alone is not evidence of tradable expectancy.

In [ ]:
concentration=tail_concentration(trades)
display(concentration.loc[(concentration.k==5)&(concentration.top_fraction==.01)].sort_values(['fold','top_share_of_net'],ascending=[True,False]).round(5))
print('Baseline and top challengers, remaining mean after excluding top 1% of realized winners:')
display(concentration.loc[(concentration.k==5)&(concentration.top_fraction==.01)&(concentration.model.isin(['ridge_all']+top)),['fold','model','top_share_of_net','rest_mean']].round(6))

## Experiments 51–53 · Stock concentration, repeated selections and calendar regime checks

In [ ]:
base=trades.loc[(trades.model=='ridge_all')&(trades.k==5)].copy()
print('Baseline largest contributors:')
display(base.groupby('symbol').agg(n=('next_return','size'),sum_return=('next_return','sum'),mean_return=('next_return','mean')).sort_values('sum_return',ascending=False).head(25).round(5))
print('Calendar-year mean return for baseline:')
base['year']=pd.to_datetime(base.bar_timestamp,utc=True).dt.year
display(base.groupby(['fold','year']).agg(n=('next_return','size'),mean_return=('next_return','mean')).round(5))

## Experiments 54–56 · Ranking skill versus tail precision at top-k

In [ ]:
rows=[]
for (fold,model,k),g in trades.groupby(['fold','model','k']):
    if k!=5:continue
    rows.append(dict(fold=fold,model=model,k=k,n=len(g),p_up3=float((g.next_return>.03).mean()),p_up5=float((g.next_return>.05).mean()),p_up10=float((g.next_return>.10).mean()),p_down5=float((g.next_return<-.05).mean()),p_down10=float((g.next_return<-.10).mean())))
tail_hit=pd.DataFrame(rows)
display(tail_hit.groupby('model',as_index=False).mean(numeric_only=True).sort_values('p_up5',ascending=False).round(5))

## Experiments 57–59 · Model search caution, positive-fold consistency and research candidate table

In [ ]:
summary=leader.loc[leader.k.eq(5)].merge(paired_summary[['model','mean_lift','positive_fold_lifts']],on='model',how='left')
summary['exploratory_candidate']=(summary.positive_folds==3)&(summary.market_excess>0)&(summary.profit_factor>1)&(summary.mean_daily>0)
display(summary.sort_values('mean_daily',ascending=False).round(6))
print('CAUTION: These are development-fold screening results after many searches, not independent p-values or proof of an edge.')

## Experiments 60–62 · Reproducible exports, audit, and next research decision

In [ ]:
save_outputs(OUT,metrics=metrics,daily=daily,trades=trades,events=ev,audit=results['audit'],paired=paired,calibration=cal,concentration=concentration,tail_hit=tail_hit,summary=summary,baseline_reconciliation=compare)
print('Saved results under',OUT)
print('Research decision: prioritize robust paired improvement, tail calibration, and consistent folds over the highest development mean.')
print('Closed holdout and prospective ledger untouched.')